# Lab 1 Solution — Projectile Motion Solver

**Physics 311: Mechanics — Modeling, Computing, Publishing**
Block 1: Foundations of Numerical Mechanics · Week 1

---

## How to use this notebook

This is *one* correct solution, not *the* correct solution. If your code looks different but
produces the same numbers, your code is fine. Read this for three things:

1. **The numbers.** Compare your range, landing time, and optimal angles against the values here.
   Small differences (< 0.1%) are step-size effects. Large differences are bugs worth finding.
2. **The structure.** Notice how one `simulate_projectile()` function serves Part B *and* the angle
   sweep, and how Part C reuses the same skeleton with one changed line. That pattern —
   *write the loop once, change only the force* — is the whole architectural idea of this course.
3. **The written answers.** Every part of this lab asked you to interpret a plot, not just produce
   it. The `Discussion` cells below show the depth expected.

> **A note on the interpolation trick in `find_landing()`:** many submissions used the last stored
> point (`x[-1]`) as the range. That point is *below* ground, so it carries a systematic
> step-size-sized error that changes with launch angle. If your range–angle curve came out slightly
> asymmetric about 45°, this was almost certainly why. It is worth understanding.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Physical constants
G = 9.81          # m/s^2, magnitude of gravitational acceleration

# Plot styling (UT orange for emphasis)
UT_ORANGE = "#ff8200"
UT_SMOKEY = "#58595b"
plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("NumPy      :", np.__version__)
print("Matplotlib :", plt.matplotlib.__version__)

---
# Part A — One-Dimensional Free Fall (20 points)

A ball is released from rest at $h = 100$ m and falls under uniform gravity, no air resistance.

**State vector:** $\mathbf{S} = [y,\ v_y]$, with $+y$ measured **upward**.

**Physics:** the only force is gravity, so $a = -g$ (constant).

**Exact solution** (our answer key for this part):

$$y(t) = h - \tfrac{1}{2}gt^2, \qquad v(t) = -gt, \qquad t_f = \sqrt{2h/g}$$

### A1 — The simulator

Three things to notice in the implementation below:

- The `while y > 0` loop runs until the ball passes ground level. We do **not** know the number of
  steps in advance, which is why this is a `while` and not a `for`.
- Positions and velocities are appended to Python lists and converted to arrays *once* at the end.
  Appending to a NumPy array inside a loop reallocates memory every step and is far slower.
- `simulate_freefall` takes `dt` as an argument rather than hard-coding it. That single decision is
  what makes the step-size study in A3 a three-line loop instead of five copy-pasted blocks.

In [ ]:
def simulate_freefall(h, dt, g=G):
    '''
    Simulate 1D free fall from rest using the Euler method.

    Parameters
    ----------
    h  : initial height (m), released from rest
    dt : time step (s)
    g  : gravitational acceleration magnitude (m/s^2)

    Returns
    -------
    t, y, vy : 1D arrays of time, height, and vertical velocity
    '''
    y, vy, t = h, 0.0, 0.0
    ts, ys, vys = [t], [y], [vy]

    while y > 0:
        a  = -g              # the physics: one line
        y  = y + vy * dt     # position update uses the OLD velocity (this is Euler)
        vy = vy + a * dt     # velocity update
        t  = t + dt
        ts.append(t); ys.append(y); vys.append(vy)

    return np.array(ts), np.array(ys), np.array(vys)


# Exact solution, for validation
def exact_freefall(t, h, g=G):
    return h - 0.5 * g * t**2, -g * t

### A2 — Compare against the exact solution

With $h = 100$ m and $g = 9.81$ m/s², theory gives

$$t_f = \sqrt{2(100)/9.81} = 4.5152\ \text{s}, \qquad v_{\rm impact} = -g t_f = -44.29\ \text{m/s}$$

In [ ]:
h  = 100.0
dt = 0.01

t, y, vy = simulate_freefall(h, dt)

t_exact = np.sqrt(2 * h / G)
v_exact = -G * t_exact

print(f"Exact landing time     : {t_exact:.4f} s")
print(f"Simulated landing time : {t[-1]:.4f} s   (last stored step, y = {y[-1]:.3f} m)")
print(f"Exact impact speed     : {v_exact:.3f} m/s")
print(f"Simulated impact speed : {vy[-1]:.3f} m/s")
print(f"Number of steps        : {len(t)}")

In [ ]:
y_th, v_th = exact_freefall(t, h)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(t, y_th, "k-", lw=2.5, label="Exact  $h - \\frac{1}{2}gt^2$")
ax[0].plot(t, y, "--", color=UT_ORANGE, lw=2, label=f"Euler, dt = {dt} s")
ax[0].axhline(0, color="gray", lw=1)
ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("height (m)")
ax[0].set_title("Free fall from 100 m"); ax[0].legend()

ax[1].plot(t, y - y_th, color=UT_ORANGE, lw=2)
ax[1].axhline(0, color="gray", lw=1)
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("$y_{\\rm sim} - y_{\\rm exact}$ (m)")
ax[1].set_title("Euler position error grows with time")

plt.tight_layout(); plt.show()

The two curves are visually indistinguishable, but the residual plot on the right shows the truth:
Euler's position is *systematically high*, and the error grows steadily. That is not random noise —
it is the neglected $\tfrac{1}{2}g\,\Delta t^2$ term accumulating once per step.

We will make that statement quantitative in Week 2. For now, note the sign: **Euler makes the ball
fall too slowly.**

### A3 — Step-size study, and the landing-time subtlety

The loop exits on the first step where $y < 0$, so the last stored point is already *underground*.
Reporting `t[-1]` therefore overestimates the landing time by up to one full $\Delta t$ — an error
that has nothing to do with the integrator.

The fix is to interpolate linearly between the last two stored points to find the instant $y = 0$:

In [ ]:
def landing_time(t, y):
    '''Linearly interpolate the time at which y crosses zero.'''
    frac = y[-2] / (y[-2] - y[-1])       # fraction of the final step needed to reach y = 0
    return t[-2] + frac * (t[-1] - t[-2])


print(f"{'dt (s)':>8} {'steps':>8} {'t[-1]':>10} {'interpolated':>14} {'error':>12}")
print("-" * 56)
for dt_test in [0.1, 0.05, 0.01, 0.005, 0.001]:
    tt, yy, vv = simulate_freefall(h, dt_test)
    t_land = landing_time(tt, yy)
    print(f"{dt_test:>8} {len(tt):>8} {tt[-1]:>10.4f} {t_land:>14.6f} {t_land - t_exact:>12.2e}")

Two separate effects are visible in that table, and telling them apart is the point of the exercise:

| Column | What it measures |
|---|---|
| `t[-1]` | Integrator error **+** the crude "first step below zero" landing detection |
| `interpolated` | Integrator error alone |

The interpolated error halves cleanly when $\Delta t$ halves — the signature of a **first-order**
method. Let us confirm that by measuring the slope on a log–log plot.

In [ ]:
dts  = np.array([0.1, 0.05, 0.025, 0.0125, 0.00625])
errs = np.array([abs(landing_time(*simulate_freefall(h, d)[:2]) - t_exact) for d in dts])

order = np.polyfit(np.log(dts), np.log(errs), 1)[0]

print("successive error ratios :", np.round(errs[:-1] / errs[1:], 3))
print(f"measured order of accuracy : {order:.3f}")

plt.figure(figsize=(5.5, 4))
plt.loglog(dts, errs, "o-", color=UT_ORANGE, lw=2, label=f"measured (slope = {order:.2f})")
plt.loglog(dts, errs[0] * (dts / dts[0]), "k--", lw=1.5, label="reference slope 1")
plt.xlabel("$\\Delta t$ (s)"); plt.ylabel("landing-time error (s)")
plt.title("Euler is first-order accurate"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

> **Discussion (Part A).**
> The Euler method reproduces the free-fall trajectory well enough to eyeball, but the error is
> systematic rather than random: simulated height is always *too high*, because the position update
> uses the velocity from the *start* of the interval, and the ball is speeding up throughout it.
> Every step therefore under-counts the distance fallen by $\tfrac{1}{2}g\Delta t^2$.
>
> Measured across five step sizes, the landing-time error scales as $\Delta t^{1.00}$ — first order,
> exactly as the truncated Taylor series predicts. Halving $\Delta t$ buys only a factor-of-two
> accuracy improvement at double the cost, which is a poor exchange rate. This is precisely the
> motivation for the better integrators in Weeks 2 and 5.
>
> Separately, the naive landing time `t[-1]` is biased high by up to one full $\Delta t$ regardless
> of integrator quality. Interpolating to the $y = 0$ crossing removes that bias and is essential
> before any range measurement can be trusted.

---
# Part B — Two-Dimensional Projectile (35 points)

Now four state variables: $\mathbf{S} = [x,\ y,\ v_x,\ v_y]$, launched at speed $v_0 = 25$ m/s and
angle $\theta$ from the origin.

**Physics:** gravity acts only downward, so $a_x = 0$ and $a_y = -g$. The horizontal and vertical
motions are completely independent — which is the whole reason the closed-form solution exists.

**Analytic results for a level launch and landing:**

$$R = \frac{v_0^2 \sin 2\theta}{g}, \qquad
H = \frac{(v_0\sin\theta)^2}{2g}, \qquad
T = \frac{2 v_0 \sin\theta}{g}$$

### B1 — The 2D simulator

Compare this function to `simulate_freefall`. The loop body is the *same three ideas* — compute
acceleration, update velocity, update position — just with two components instead of one. **Adding a
dimension did not change the method.**

One upgrade: this version uses **Euler-Cromer** (velocity updated *before* position). It costs
nothing, and Week 2 explains exactly what it buys. The `method` argument lets you switch back.

In [ ]:
def simulate_projectile(v0, angle_deg, dt=1e-3, g=G, y0=0.0, method="euler-cromer"):
    '''
    Simulate 2D projectile motion with no air resistance.

    Returns
    -------
    t, x, y, vx, vy : 1D arrays
    '''
    theta = np.radians(angle_deg)
    x, y   = 0.0, y0
    vx, vy = v0 * np.cos(theta), v0 * np.sin(theta)

    ts, xs, ys, vxs, vys = [0.0], [x], [y], [vx], [vy]
    t = 0.0

    while y >= 0.0:
        ax, ay = 0.0, -g                 # the physics: gravity only

        if method == "euler":
            x_new = x + vx * dt          # positions use OLD velocities
            y_new = y + vy * dt
            vx, vy = vx + ax * dt, vy + ay * dt
            x, y = x_new, y_new
        else:                            # euler-cromer
            vx, vy = vx + ax * dt, vy + ay * dt
            x, y = x + vx * dt, y + vy * dt

        t += dt
        ts.append(t); xs.append(x); ys.append(y); vxs.append(vx); vys.append(vy)

    return (np.array(ts), np.array(xs), np.array(ys),
            np.array(vxs), np.array(vys))


def find_range(x, y):
    '''Interpolate the horizontal position at the instant y = 0.'''
    frac = y[-2] / (y[-2] - y[-1])
    return x[-2] + frac * (x[-1] - x[-2])

### B2 — Validate a single trajectory at 45°

In [ ]:
v0 = 25.0
t, x, y, vx, vy = simulate_projectile(v0, 45.0)

R_sim, H_sim, T_sim = find_range(x, y), y.max(), t[-1]
theta = np.radians(45.0)
R_th = v0**2 * np.sin(2 * theta) / G
H_th = (v0 * np.sin(theta))**2 / (2 * G)
T_th = 2 * v0 * np.sin(theta) / G

print(f"{'quantity':<14}{'simulated':>12}{'theory':>12}{'error':>10}")
print("-" * 48)
for name, s, th, unit in [("range",  R_sim, R_th, "m"),
                          ("max height", H_sim, H_th, "m"),
                          ("flight time", T_sim, T_th, "s")]:
    print(f"{name:<14}{s:>12.4f}{th:>12.4f}{100*abs(s-th)/th:>9.3f}%")

All three agree with theory to better than 0.05% at $\Delta t = 10^{-3}$ s. That is the standard to
hold yourself to *before* moving to a problem with no answer key — which is exactly what Part C is.

### B3 — The angle sweep

This is the heart of Part B, and the structural point is worth stating plainly: the sweep is a
**loop around a function that has already been validated**. No new physics, no new simulation code.

If you found yourself writing a fresh simulation loop for this task, the signal is that
`simulate_projectile` was not sufficiently parameterized.

In [ ]:
angles = np.arange(10, 85, 5)          # 10, 15, ..., 80 degrees
ranges = np.array([find_range(*simulate_projectile(v0, a)[1:3]) for a in angles])

ranges_th = v0**2 * np.sin(2 * np.radians(angles)) / G

print(f"{'angle':>6}{'R_sim (m)':>12}{'R_theory (m)':>14}{'error':>10}")
print("-" * 44)
for a, r, rt in zip(angles, ranges, ranges_th):
    print(f"{a:>6}{r:>12.4f}{rt:>14.4f}{100*abs(r-rt)/rt:>9.3f}%")

i_max = np.argmax(ranges)
print(f"\nMaximum range on this grid : {ranges[i_max]:.3f} m at {angles[i_max]}deg")
print(f"Theoretical maximum        : {v0**2/G:.3f} m at 45deg")

In [ ]:
theta_fine = np.linspace(0, 90, 400)

plt.figure(figsize=(7.5, 4.5))
plt.plot(theta_fine, v0**2 * np.sin(2 * np.radians(theta_fine)) / G,
         "k-", lw=2, label="theory  $v_0^2\\sin 2\\theta / g$")
plt.plot(angles, ranges, "o", color=UT_ORANGE, ms=8, label="simulation")
plt.axvline(45, color=UT_SMOKEY, ls=":", lw=1.5, label="45deg")
plt.xlabel("launch angle (degrees)"); plt.ylabel("range (m)")
plt.title(f"Range vs. launch angle, $v_0$ = {v0:.0f} m/s, no drag")
plt.legend(); plt.tight_layout(); plt.show()

### B4 — Two things the coarse sweep hides

**(i) The grid found 45° by luck, not by measurement.** Because 45° happens to lie exactly on a 5°
grid, `argmax` returns the right answer for the wrong reason. A 5° grid can only ever locate an
optimum to $\pm 2.5°$. This matters enormously in Part C, where the true optimum is *not* on the grid.

**(ii) The peak is remarkably flat.** Let us quantify both with a 1° sweep.

In [ ]:
fine  = np.arange(30.0, 61.0, 1.0)
R_fine = np.array([find_range(*simulate_projectile(v0, a)[1:3]) for a in fine])

R_45 = R_fine[fine == 45.0][0]
R_40 = R_fine[fine == 40.0][0]
R_35 = R_fine[fine == 35.0][0]

print(f"1-degree sweep optimum : {fine[np.argmax(R_fine)]:.0f}deg")
print(f"R(45) = {R_45:.3f} m")
print(f"R(40) = {R_40:.3f} m   ->  {100*(R_45-R_40)/R_45:.2f}% below peak")
print(f"R(35) = {R_35:.3f} m   ->  {100*(R_45-R_35)/R_45:.2f}% below peak")
print("\nSymmetry check (theory predicts R(theta) = R(90-theta)):")
for a in [30.0, 35.0, 40.0]:
    ra = R_fine[fine == a][0]
    rb = R_fine[fine == 90.0 - a][0]
    print(f"  R({a:.0f}) = {ra:.4f}   R({90-a:.0f}) = {rb:.4f}   difference = {abs(ra-rb):.2e} m")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for a, style in [(30, ":"), (45, "-"), (60, "--")]:
    tt, xx, yy, _, _ = simulate_projectile(v0, a)
    ax[0].plot(xx, yy, style, lw=2, label=f"{a}deg")
ax[0].axhline(0, color="gray", lw=1)
ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("y (m)")
ax[0].set_title("30deg and 60deg land in the same place"); ax[0].legend(); ax[0].set_aspect("equal")

ax[1].plot(fine, R_fine, "-", color=UT_ORANGE, lw=2)
ax[1].axvline(45, color=UT_SMOKEY, ls=":", lw=1.5)
ax[1].axhline(0.99 * R_fine.max(), color="gray", ls="--", lw=1, label="99% of peak")
ax[1].set_xlabel("launch angle (degrees)"); ax[1].set_ylabel("range (m)")
ax[1].set_title("The peak is flat: 41deg-49deg all within 1%"); ax[1].legend()

plt.tight_layout(); plt.show()

> **Discussion (Part B).**
> The simulated range–angle curve matches $R = v_0^2\sin(2\theta)/g$ to within 0.11% at every angle
> tested, and the maximum occurs at 45° with $R_{\max} = 63.69$ m against a predicted
> $v_0^2/g = 63.71$ m. The curve is symmetric about 45° — complementary angles agree to better than
> $10^{-2}$ m — which is the $\sin 2\theta$ dependence appearing directly in the data.
>
> Two honest caveats about the method:
>
> First, a 5° grid cannot *locate* an optimum; it can only report the best sampled point. That 45°
> came out correct is a consequence of 45° lying on the grid, not evidence that the search resolved
> it. Re-running at 1° resolution confirms 45° independently.
>
> Second, the maximum is very flat: every angle from 41° to 49° lands within 1% of peak range, and
> even 35° reaches 94%. Physically, raising the angle trades horizontal speed for flight time, and
> near 45° those effects nearly cancel. This flatness is why the optimum moves so readily once drag
> is introduced in Part C — only a small asymmetry is needed to shift it several degrees.

---
# Part C — Quadratic Air Drag (35 points)

**This is the part of the lab with no answer key**, and it is the reason the course exists.

For a baseball-sized object at everyday speeds, drag opposes the velocity and scales as speed
*squared*:

$$\mathbf{F}_{\rm drag} = -k\,|\mathbf{v}|\,\mathbf{v}, \qquad k = \tfrac{1}{2}C_d \rho A$$

so the acceleration components become

$$a_x = -\frac{k}{m}\,|\mathbf{v}|\,v_x, \qquad
a_y = -g - \frac{k}{m}\,|\mathbf{v}|\,v_y, \qquad
|\mathbf{v}| = \sqrt{v_x^2 + v_y^2}$$

**Why this cannot be solved by hand:** the $|\mathbf{v}|$ factor couples the two components. The
horizontal equation now contains $v_y$ and vice versa, so the clean independence that produced
$R = v_0^2\sin 2\theta / g$ is gone. There is no elementary closed-form solution. Numerically,
however, it is a *two-line change*.

In [ ]:
# Baseball parameters
m_ball  = 0.145      # kg
r_ball  = 0.0366     # m
A_ball  = np.pi * r_ball**2
C_d     = 0.35
rho_air = 1.225      # kg/m^3

k_drag = 0.5 * C_d * rho_air * A_ball

print(f"cross-sectional area A : {A_ball:.6f} m^2")
print(f"drag coefficient k     : {k_drag:.6f} kg/m")
print(f"k/m                    : {k_drag/m_ball:.6f} 1/m")
print(f"terminal speed sqrt(mg/k) : {np.sqrt(m_ball*G/k_drag):.2f} m/s")

That terminal speed of about 40 m/s is the number to keep in mind. It tells us drag will be a
**minor** correction for a 25 m/s launch and a **dominant** one at 45 m/s, because the drag force
scales as $(v/v_{\rm term})^2$ relative to gravity.

In [ ]:
def simulate_drag(v0, angle_deg, dt=1e-3, k=k_drag, m=m_ball, g=G, y0=0.0):
    '''
    2D projectile with quadratic drag.  Set k = 0 to recover the drag-free case.
    '''
    theta = np.radians(angle_deg)
    x, y   = 0.0, y0
    vx, vy = v0 * np.cos(theta), v0 * np.sin(theta)

    ts, xs, ys, speeds = [0.0], [x], [y], [v0]
    t = 0.0

    while y >= 0.0:
        v  = np.hypot(vx, vy)              # speed magnitude couples the components
        ax = -(k / m) * v * vx             # <-- the only physics change from Part B
        ay = -g - (k / m) * v * vy

        vx, vy = vx + ax * dt, vy + ay * dt
        x,  y  = x + vx * dt,  y + vy * dt
        t += dt

        ts.append(t); xs.append(x); ys.append(y); speeds.append(np.hypot(vx, vy))

    return (np.array(ts), np.array(xs), np.array(ys), np.array(speeds))

### C1 — How much does drag actually matter?

In [ ]:
print(f"{'v0':>6}{'R no drag':>12}{'R with drag':>13}{'reduction':>11}"
      f"{'H no drag':>11}{'H drag':>9}")
print("-" * 62)
for v_test in [15.0, 25.0, 35.0, 45.0]:
    _, x1, y1, _ = simulate_drag(v_test, 45.0, k=0.0)
    _, x2, y2, _ = simulate_drag(v_test, 45.0)
    R1, R2 = find_range(x1, y1), find_range(x2, y2)
    print(f"{v_test:>6.0f}{R1:>12.2f}{R2:>13.2f}{100*(1-R2/R1):>10.1f}%"
          f"{y1.max():>11.2f}{y2.max():>9.2f}")

In [ ]:
v0_c = 25.0
t_nd, x_nd, y_nd, s_nd = simulate_drag(v0_c, 45.0, k=0.0)
t_d,  x_d,  y_d,  s_d  = simulate_drag(v0_c, 45.0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(x_nd, y_nd, "k-",  lw=2, label=f"no drag (R = {find_range(x_nd,y_nd):.1f} m)")
ax[0].plot(x_d,  y_d,  "-", color=UT_ORANGE, lw=2,
           label=f"quadratic drag (R = {find_range(x_d,y_d):.1f} m)")
ax[0].axhline(0, color="gray", lw=1)
ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("y (m)")
ax[0].set_title(f"Trajectory at 45deg, $v_0$ = {v0_c:.0f} m/s"); ax[0].legend()
ax[0].set_aspect("equal")

ax[1].plot(t_nd, s_nd, "k-", lw=2, label="no drag")
ax[1].plot(t_d,  s_d,  "-", color=UT_ORANGE, lw=2, label="with drag")
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("speed $|v|$ (m/s)")
ax[1].set_title("Drag removes speed monotonically"); ax[1].legend()

plt.tight_layout(); plt.show()

Two features of the drag trajectory are worth naming explicitly, because they are the physical
signature of a velocity-dependent force:

1. **It is asymmetric.** The descending branch is steeper than the ascending one. The projectile
   loses horizontal speed throughout the flight, so it cannot travel as far coming down as it did
   going up. A drag-free parabola is perfectly symmetric; this is not a parabola at all.
2. **The speed curve no longer returns to its launch value.** Without drag, the ball lands at
   exactly $v_0$ (energy conservation). With drag, kinetic energy has been irreversibly converted to
   heat in the air, so it lands slower. That distinction — *physical* energy loss versus *numerical*
   energy drift — is the entire subject of Week 2.

### C2 — Where is the optimal angle now?

Section B4 established the discipline required here: a coarse grid cannot resolve an optimum. With
drag, the true optimum is *not* on a 5° grid, so we sweep coarse first, then refine.

In [ ]:
for v_test in [25.0, 45.0]:
    coarse = np.arange(10.0, 85.0, 5.0)
    Rc = np.array([find_range(*simulate_drag(v_test, a)[1:3]) for a in coarse])

    fine_c = np.arange(25.0, 56.0, 1.0)
    Rf = np.array([find_range(*simulate_drag(v_test, a)[1:3]) for a in fine_c])

    print(f"v0 = {v_test:.0f} m/s")
    print(f"   5-degree grid : optimum {coarse[np.argmax(Rc)]:.0f}deg, R = {Rc.max():.2f} m")
    print(f"   1-degree grid : optimum {fine_c[np.argmax(Rf)]:.0f}deg, R = {Rf.max():.2f} m")
    print(f"   drag-free optimum is 45deg, R = {v_test**2/G:.2f} m\n")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
for v_test, col in [(25.0, UT_ORANGE), (45.0, UT_SMOKEY)]:
    ang = np.arange(20.0, 71.0, 1.0)
    Rd  = np.array([find_range(*simulate_drag(v_test, a)[1:3]) for a in ang])
    Rn  = v_test**2 * np.sin(2 * np.radians(ang)) / G
    plt.plot(ang, Rn / Rn.max(), "--", color=col, lw=1.5, alpha=0.6,
             label=f"$v_0$={v_test:.0f}, no drag")
    plt.plot(ang, Rd / Rd.max(), "-", color=col, lw=2.5,
             label=f"$v_0$={v_test:.0f}, with drag (peak {ang[np.argmax(Rd)]:.0f}deg)")
plt.axvline(45, color="k", ls=":", lw=1.2)
plt.xlabel("launch angle (degrees)"); plt.ylabel("range / max range")
plt.title("Drag shifts the optimal angle below 45deg, more so at higher speed")
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

### C3 — Is the answer converged?

A result is only trustworthy if it stops changing when $\Delta t$ shrinks. Since there is no exact
solution to compare against, self-consistency is the *only* available check.

In [ ]:
print(f"{'dt (s)':>10}{'R at 40deg (m)':>18}{'change from previous':>22}")
print("-" * 52)
prev = None
for d in [0.02, 0.01, 0.005, 0.001, 0.0005]:
    R = find_range(*simulate_drag(45.0, 40.0, dt=d)[1:3])
    change = "" if prev is None else f"{R - prev:+.5f} m"
    print(f"{d:>10}{R:>18.5f}{change:>22}")
    prev = R

print("\nOptimum angle vs. step size:")
for d in [0.01, 0.001]:
    ang = np.arange(35.0, 46.0, 1.0)
    R = [find_range(*simulate_drag(45.0, a, dt=d)[1:3]) for a in ang]
    print(f"   dt = {d:<8} optimum = {ang[int(np.argmax(R))]:.0f}deg")

> **Discussion (Part C).**
> Quadratic drag reduces the 45° range of a baseball from 63.69 m to 49.22 m at $v_0 = 25$ m/s — a
> 23% loss — and from 206.4 m to 110.0 m at $v_0 = 45$ m/s, a 47% loss. The reduction grows sharply
> with launch speed because drag scales as $v^2$ while gravity does not; the ratio of the two forces
> is roughly $(v/v_{\rm term})^2$, and $v_{\rm term} \approx 40$ m/s for this ball.
>
> **The optimal angle drops below 45°:** to 43° at $v_0 = 25$ m/s and to 40° at $v_0 = 45$ m/s. The
> physical reason is that drag penalizes *time aloft*. A high launch angle buys range in the
> drag-free case by keeping the projectile in flight longer, but with drag every extra second is
> another second of horizontal speed being eroded. The optimum therefore shifts toward a flatter,
> faster, shorter trajectory, and shifts further the more drag matters. Because the drag-free peak is
> so flat (Part B4), only a small asymmetry is needed to move the optimum several degrees.
>
> The trajectory itself is no longer a parabola: the descending branch is visibly steeper than the
> ascending one, and the impact speed is below the launch speed because kinetic energy has been
> dissipated as heat.
>
> On numerical reliability: the range at 40° changes by only 0.02% between $\Delta t = 10^{-3}$ and
> $5\times10^{-4}$ s, and the reported optimum is unchanged from $\Delta t = 10^{-2}$ down to
> $10^{-3}$ s. The result is converged at the precision quoted. Since no analytic solution exists,
> this self-consistency check is the only validation available — which makes it mandatory, not
> optional.

---
# Bonus — The Home Run Problem (+10 points)

**Question:** a batter makes contact 1 m above the ground. The outfield fence is 120 m away and 3 m
high. Can the ball clear it?

This question cannot be answered analytically, and it cannot be answered by a single simulation
either — it requires a *sweep* over launch angle at each speed. This is the pattern used repeatedly
for the rest of the course.

In [ ]:
FENCE_X, FENCE_H, Y0 = 120.0, 3.0, 1.0

def height_at_fence(v0, angle_deg, fence_x=FENCE_X, y0=Y0, dt=1e-3):
    '''Height of the ball as it crosses the fence line; NaN if it never gets there.'''
    _, x, y, _ = simulate_drag(v0, angle_deg, dt=dt, y0=y0)
    if x[-1] < fence_x:
        return np.nan                      # landed short of the fence
    return np.interp(fence_x, x, y)


def max_distance(v0, dt=1e-3):
    '''Best range over all launch angles, used when the ball never reaches the fence.'''
    angs = np.arange(10.0, 61.0, 1.0)
    return max(find_range(*simulate_drag(v0, a, dt=dt, y0=Y0)[1:3]) for a in angs)


print(f"Fence: {FENCE_X:.0f} m away, {FENCE_H:.0f} m high.  Contact at {Y0:.0f} m.\n")
for v_test in [40.0, 45.0, 50.0, 55.0]:
    angs = np.arange(10.0, 61.0, 1.0)
    hs   = np.array([height_at_fence(v_test, a) for a in angs])

    if np.all(np.isnan(hs)):                       # never even reaches the fence line
        print(f"v0 = {v_test:.0f} m/s : short at every angle "
              f"| longest drive {max_distance(v_test):.1f} m")
        continue

    best = int(np.nanargmax(hs))
    ok   = angs[np.nan_to_num(hs, nan=-99.0) > FENCE_H]
    if len(ok):
        print(f"v0 = {v_test:.0f} m/s : CLEARS for {ok.min():.0f}deg-{ok.max():.0f}deg "
              f"| best {angs[best]:.0f}deg, {hs[best]:.2f} m at the wall")
    else:
        print(f"v0 = {v_test:.0f} m/s : falls short "
              f"| best {angs[best]:.0f}deg reaches {hs[best]:.2f} m at the wall")

In [ ]:
# Minimum launch speed needed, to 0.1 m/s
lo, hi = 40.0, 60.0
for _ in range(40):
    mid = 0.5 * (lo + hi)
    hs = np.array([height_at_fence(mid, a) for a in np.arange(20.0, 56.0, 1.0)])
    best_h = -99.0 if np.all(np.isnan(hs)) else np.nanmax(hs)
    if best_h > FENCE_H:
        hi = mid
    else:
        lo = mid
print(f"Minimum launch speed to clear the fence : {hi:.1f} m/s  ({hi*2.237:.0f} mph)")

angs = np.arange(20.0, 56.0, 1.0)
hs   = [height_at_fence(hi + 0.5, a) for a in angs]
print(f"Optimal angle at that speed             : {angs[int(np.nanargmax(hs))]:.0f}deg")

In [ ]:
plt.figure(figsize=(8, 4.5))
for v_test, style in [(45.0, "--"), (50.0, "-"), (55.0, "-")]:
    _, x, y, _ = simulate_drag(v_test, 38.0, y0=Y0)
    plt.plot(x, y, style, lw=2, label=f"$v_0$ = {v_test:.0f} m/s at 38deg")

plt.plot([FENCE_X, FENCE_X], [0, FENCE_H], color=UT_ORANGE, lw=6,
         solid_capstyle="butt", label="fence (3 m)")
plt.axhline(0, color="gray", lw=1)
plt.xlabel("distance from home plate (m)"); plt.ylabel("height (m)")
plt.title("Clearing a 120 m fence requires about 49 m/s off the bat")
plt.legend(); plt.xlim(0, 150); plt.ylim(0, 40)
plt.tight_layout(); plt.show()

> **Discussion (Bonus).**
> With realistic drag, a ball struck at 45 m/s cannot clear a 120 m fence at any launch angle — its
> best case falls essentially at the wall. The minimum required launch speed is about **49 m/s
> (roughly 109 mph)**, achieved near a 41° launch angle. At 50 m/s the ball clears for launch
> angles from about 32° to 48°, so the window is generous once the speed threshold is met.
>
> This is a satisfying sanity check against reality: measured exit velocities for major-league home
> runs cluster around 45–55 m/s (100–120 mph), and typical home-run distances are 110–140 m.
> Our simple two-parameter drag model lands squarely in the right regime.
>
> It is worth noting what the model still omits — most importantly **backspin**, which generates
> aerodynamic lift (the Magnus force) and adds roughly 10–20 m of carry to a well-struck ball. Adding
> a velocity-dependent transverse force would be a natural extension, and it is exactly the kind of
> new `Force` object the engine architecture in Week 7 is designed to accommodate.

---
# What the graders were looking for

| Part | Most common point loss | How to avoid it |
|---|---|---|
| A | Reporting `t[-1]` as the landing time without comment | Interpolate to the $y=0$ crossing, or explicitly note the one-$\Delta t$ bias |
| A | "Euler is inaccurate" with no number attached | Quote the measured error and its scaling with $\Delta t$ |
| B | Range–angle curve slightly asymmetric about 45° | Same interpolation issue, now angle-dependent |
| B | Declaring 45° optimal from a 5° grid without caveat | State the $\pm 2.5°$ resolution limit, or refine the grid |
| C | Applying drag only to $v_y$, or forgetting the $\lvert v\rvert$ factor | Both components need $-(k/m)\lvert v\rvert v_i$ |
| C | Reporting an optimum without a step-size check | Halve $\Delta t$ and confirm the answer is stable |
| All | Plots with no axis labels or units | Label every axis; a plot without units is not a result |
| All | Notebook that fails on **Restart & Run All** | Always re-run cleanly before submitting |

## The three ideas to carry into Week 2

1. **The loop structure never changed.** Free fall, 2D projectile, and quadratic drag all used the
   identical skeleton: compute acceleration, update velocity, update position, store, repeat. Only
   the acceleration line differed. This is why a general physics engine is possible at all, and it is
   what Block 2 formalizes.

2. **Euler's error is systematic, not random,** and it is only first-order. That is the entire
   motivation for Euler-Cromer (Week 2) and Velocity Verlet (Week 5).

3. **Validation is a skill, not a formality.** In Part B you had an answer key. In Part C you did
   not — and you still had to justify your answer, using convergence, symmetry, and physical
   reasoning. Week 2 gives you the most powerful tool for this: **conservation laws as diagnostics**,
   which work even when no exact solution exists.

---

*Physics 311 · Lab 1 Solution · Block 1, Week 1*